# Macro Math Visualizer (Weekly Synthesis)
This dashboard visually unpacks the quantitative math engines over the Weekly timeframe reporting cadence. It calculates the Regime Persistence, Fragility Index, and Epistemic Kelly Exposure, and then presents the most recently generated Weekly report layout.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from train_models import fetch_training_data
import joblib
import warnings
import glob
import os
from IPython.display import Markdown
warnings.filterwarnings('ignore')

plt.style.use('dark_background')


## 1. Data Ingestion & HMM Decoder

In [ ]:
# Load the 2-year data using the exact daily training schema
df = fetch_training_data(years=2)

# Load the HMM
package = joblib.load('../models/hmm_model.pkl')
hmm = package['hmm']
scaler = package['scaler']
state_labels = package['state_labels']
features = package['feature_names']

# Predict the Hidden States
X_scaled = scaler.transform(df[features].values)
states = hmm.predict(X_scaled)
df['Regime_ID'] = states
df['Regime_Label'] = df['Regime_ID'].map(state_labels)


## 2. Regime Overlay on the S&P 500

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(df.index, df['Close'], color='white', linewidth=1.5, label='SPX')
ax.set_title('S&P 500 Price with Hidden Markov Model Regime Overlay', fontsize=16)

# Create color map for regimes
unique_regimes = df['Regime_Label'].unique()
colors = sns.color_palette("husl", len(unique_regimes))
regime_colors = dict(zip(unique_regimes, colors))

# Overlay regimes
for i in range(len(df)-1):
    regime = df['Regime_Label'].iloc[i]
    ax.axvspan(df.index[i], df.index[i+1], color=regime_colors[regime], alpha=0.3, lw=0)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=regime_colors[reg], alpha=0.5, label=reg) for reg in unique_regimes]
ax.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()


## 3. The Fragility Index (Hidden Liquidity Shocks)

In [ ]:
spx_dxy_corr = df['spx_ret'].rolling(10).corr(df['dxy_ret'])

fig, ax1 = plt.subplots(figsize=(15, 5))
ax1.plot(df.index, spx_dxy_corr, color='cyan', label='10-Day SPX/DXY Correlation')
ax1.axhline(0.4, color='red', linestyle='--', label='Fragility Threshold (>0.4)')
ax1.set_ylabel('Correlation')
ax1.set_title('Hidden Market Fragility: Dollar Liquidity Drain', fontsize=14)
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(df.index, df['vix_zscore'], color='magenta', alpha=0.5, label='VIX Z-Score')
ax2.set_ylabel('VIX Z-Score (Vol-of-Vol proxy)')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()


## 4. Epistemic Kelly Exposure Sizing

In [ ]:
kelly_exposure = np.where(df['vix_zscore'] > 2.0, 0.05, 0.5)
kelly_exposure = np.where(df['Regime_Label'].str.contains('RISK_ON'), kelly_exposure + 0.3, kelly_exposure)
kelly_exposure = np.where(df['Regime_Label'].str.contains('FEAR|SHOCK|STRESS'), 0.0, kelly_exposure)
kelly_smoothed = pd.Series(kelly_exposure).rolling(5).mean().fillna(0)

fig, ax = plt.subplots(figsize=(15, 4))
ax.fill_between(df.index, kelly_smoothed * 100, color='lime', alpha=0.4)
ax.plot(df.index, kelly_smoothed * 100, color='lime', label='Target Portfolio Exposure %')
ax.set_ylim(0, 100)
ax.set_title('Epistemic Kelly Sizing Curve (Risk Scaler)', fontsize=14)
ax.set_ylabel('Exposure %')
ax.legend()
plt.tight_layout()
plt.show()


## 5. Generated Weekly Report Layout
Displaying the most recent algorithmically generated report.


In [ ]:
reports_dir = '../reports'
pattern = 'macro weekly synthesis*.md'
files = glob.glob(os.path.join(reports_dir, pattern))

if files:
    latest_file = max(files, key=os.path.getmtime)
    with open(latest_file, 'r') as f:
        content = f.read()
    display(Markdown(content))
else:
    print("No recent report found. Run the Weekly script first.")
